# Cerebra — TRIBE v2 worker on Colab

Runs the heavy `facebook/tribev2` inference worker on Colab's free GPU and exposes it with a public Cloudflare tunnel URL. Paste that URL into your **local** `.env.local` as `TRIBEV2_API_URL`, then run `npm run dev` locally.

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is fine).

## 1. Confirm GPU

In [ ]:
!nvidia-smi -L || print('No GPU! Set Runtime > Change runtime type > GPU, then rerun.')
import sys; print('Python', sys.version)

## 2. Set your Hugging Face token
Needs read access + accepted access to the gated `meta-llama/Llama-3.2-3B`.

In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_PASTE_YOUR_TOKEN_HERE'  # <-- paste your token
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']

# Quick sanity check: token valid + gated model access
import urllib.request, json
req = urllib.request.Request('https://huggingface.co/api/models/meta-llama/Llama-3.2-3B',
                             headers={'Authorization': f"Bearer {os.environ['HF_TOKEN']}"})
try:
    urllib.request.urlopen(req); print('Token OK and Llama-3.2-3B access granted.')
except urllib.error.HTTPError as e:
    print('Problem:', e.code, '- 401=bad token, 403=request access to meta-llama/Llama-3.2-3B first')

## 3. Clone the repo and install the worker dependencies
First run takes several minutes (TRIBE v2 + torch + nilearn + mne).

In [ ]:
![ -d Cerebra ] || git clone https://github.com/edrlu/Cerebra.git
%cd /content/Cerebra
!apt-get -qq install -y ffmpeg >/dev/null
!pip install -q -r worker/requirements.txt

## 4. Download the Cloudflare tunnel binary

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

## 5. Start the worker + tunnel
On first run the worker downloads the model weights (several GB) before it reports ready. Wait for the **`https://....trycloudflare.com`** URL to print, then for `Worker is READY`.

In [ ]:
import subprocess, time, re, os, urllib.request

# Cache model weights on the Colab disk for this session
env = {**os.environ, 'TRIBEV2_CACHE_DIR': '/content/tribe-cache', 'CORS_ORIGINS': '*'}

worker = subprocess.Popen(
    ['uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/Cerebra/worker', env=env,
    stdout=open('/content/worker.log', 'w'), stderr=subprocess.STDOUT)
print('Worker starting (PID %d). Logs: /content/worker.log' % worker.pid)

tun = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tun.stdout:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0); break
print('\n==============================================================')
print('  PUBLIC WORKER URL:', url)
print('  Put this in your local .env.local:')
print('  TRIBEV2_API_URL=' + (url or '<failed - check tunnel>'))
print('==============================================================\n')

# Wait for the model to finish loading
for i in range(120):
    time.sleep(10)
    try:
        r = json.loads(urllib.request.urlopen('http://localhost:8000/health', timeout=5).read())
        if r.get('ready'):
            print('Worker is READY ->', r); break
        print('...loading model (%ds)' % ((i+1)*10))
    except Exception:
        print('...worker still starting (%ds)' % ((i+1)*10))

## 6. Keep this tab open
The tunnel and worker live as long as this notebook runs. If you need to see worker logs:

In [ ]:
!tail -n 40 /content/worker.log